In [1]:
from langchain_openai import ChatOpenAI
from rag_system.vector_store import VectorStoreManager
from rag_system.orchestrator import Orchestrator
from rag_system.utils import Domain
import os

from dotenv import load_dotenv
load_dotenv()

vsm = VectorStoreManager(data_dir="data/")

if os.path.exists("faiss_indices"):
    vsm.load("faiss_indices")
    print("Loaded indices from disk")
else:
    vsm.initialize()
    vsm.save("faiss_indices")
    print("Built and saved indices")
    
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)

orchestrator = Orchestrator(llm, vsm)

c:\Users\VladislavManolov\Documents\multi_agent_rag_orchestration_system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 3764.82it/s]


Loaded indices from disk


In [2]:
test_queries = [
    "What's the process for deploying a new microservice and what compliance checks are needed?",
    "How do I troubleshoot API performance issues while following our security policies?",
    "What business approvals are required for implementing a new data processing workflow?"
]

for i, query in enumerate(test_queries, 1):
    print(f"Scenario {i}: {query}")

    result = orchestrator.query(query)

    print(f"Domains: {[d.value for d in result.classification.domains]}")
    
    print("Sub-queries:")
    for domain, sub_query in result.classification.sub_queries.items():
        print(f"  {domain}: {sub_query}")
        
    print(f"Answer:\n{result.answer}\n")
    
    print(f"Citations ({len(result.citations)}):")
    for citation in result.citations:
        print(citation)
        
    print(f"{'='*80}\n")

Scenario 1: What's the process for deploying a new microservice and what compliance checks are needed?
Domains: ['technical', 'business', 'compliance']
Sub-queries:
  technical: What are the detailed steps and best practices for deploying a new microservice within our infrastructure, including CI/CD pipelines and monitoring setup?
  business: What approvals or workflow steps are required before deploying a new microservice?
  compliance: What compliance checks and security policies must be verified before and after deploying a new microservice?
Answer:
Deploying a new microservice at Postbank involves a coordinated process spanning technical deployment steps, business change management, and compliance/security verifications. Below is a comprehensive synthesis of the process and required compliance checks:

---

## 1. Technical Deployment Process

**Code Commit and Build:**
- Developers push code to the main branch in GitLab.
- CI pipeline runs unit tests, static code analysis (SonarQub

Here are some questions whose answers aren't covered in any of the three PDFs:

**Cross-domain gaps:**

- "What is the process for rotating API keys?" — you're already using this one for the knowledge update demo

- "How do I set up a new development environment locally?"

- "What is the onboarding process for new engineers joining a team?"

- "How do I request a budget for a new project?"

**Technical gaps:**

- "How do I configure auto-scaling for my microservice?"

- "What is the disaster recovery procedure if the primary data center goes down?"

- "How do I set up database replication across regions?"

**Business gaps:**

- "What is the process for hiring a contractor for a project?"

- "How do I submit an expense report?"

**Compliance gaps:**

- "What are the rules for using personal devices to access company systems?"

- "What is the policy on open-source software contributions?"

In [ ]:
# Showing that a query about a new topic returns no relevant results
result_before = orchestrator.query("What are the rules for using personal devices to access company systems?")
print("Before knowledge update:")
print(result_before.answer)
print(f"Citations: {len(result_before.citations)}\n")

for citation in result_before.citations:
    print(citation)
    
# Adding new content to the compliance domain
new_content = """
Rules for Using Personal Devices (BYOD) to Access Company Systems

1. Device Registration: Employees must register their personal devices with IT before accessing company resources.
2. Security Software: Personal devices must have up-to-date antivirus software and a secure lock screen enabled.
3. Data Access: Access to sensitive data may be restricted on personal devices. Employees must use company-approved applications for accessing corporate data.
4. Network Access: Personal devices should connect to the company network via VPN when accessing internal resources
5. Compliance: Employees must comply with all company policies regarding data security and privacy when using personal devices.
"""

chunks_added = vsm.add_document(
    domain=Domain.COMPLIANCE,
    text=new_content,
    source="byod_policy.pdf",
)
print(f"Added {chunks_added} new chunks to compliance domain\n")
print(f"{'='*80}\n")

# Same query now finds relevant content
result_after = orchestrator.query("What are the rules for using personal devices to access company systems?")
print("After knowledge update:")
print(result_after.answer)
print(f"Citations: {len(result_after.citations)}")

for citation in result_after.citations:
    print(citation)

Before knowledge update:
The provided compliance manual context does not include any specific policies or security rules governing the use of personal devices to access company systems. Therefore, I do not know the compliance policies and security rules related to personal device usage based on the given information.
Citations: 4

Added 1 new chunks to compliance domain


After knowledge update:
The compliance rules, security policies, and regulations governing the use of personal devices (BYOD) to access Postbank company systems are as follows:

1. Device Registration: Employees must register their personal devices with IT before accessing any company resources.

2. Security Software: Personal devices must have up-to-date antivirus software installed and a secure lock screen enabled to protect access.

3. Data Access: Access to sensitive data on personal devices may be restricted. Employees are required to use company-approved applications when accessing corporate data to ensure secur